# Module 15: Text Categorization


## 🗂️ What is Text Categorization (TextCat)?

Text categorization is the process of assigning a label (or multiple labels) to an **entire document**, rather than individual tokens or spans.

Examples:
- Sentiment Analysis (`POSITIVE`, `NEGATIVE`)
- Spam Detection (`SPAM`, `HAM`)
- Support Ticket Routing (`BILLING`, `TECH_SUPPORT`, `SALES`)

### Exclusive vs Non-Exclusive
spaCy supports two types of text categorization:
1. **textcat** (Exclusive): A document can only belong to ONE category. (e.g. It is either SPAM or HAM, it cannot be both).
2. **textcat_multilabel** (Non-Exclusive): A document can belong to multiple categories simultaneously. (e.g. A movie review can be both `FUNNY` and `SCARY`).


<br><br>

---

<br><br>


## 📝 Formatting TextCat Data

Unlike NER (where we provide character indices), TextCat data relies on a dictionary of boolean flags for the categories.


In [ ]:
import spacy
from spacy.tokens import DocBin

# Format: (text, {"cats": {"CATEGORY_1": 1.0, "CATEGORY_2": 0.0}})
TEXTCAT_TRAIN = [
    ("This movie was absolutely wonderful!", {"cats": {"POSITIVE": 1.0, "NEGATIVE": 0.0}}),
    ("I hated every minute of it.", {"cats": {"POSITIVE": 0.0, "NEGATIVE": 1.0}}),
    ("Best experience of my life.", {"cats": {"POSITIVE": 1.0, "NEGATIVE": 0.0}}),
    ("Do not buy this product.", {"cats": {"POSITIVE": 0.0, "NEGATIVE": 1.0}})
]

# Let's write the converter function
def convert_textcat_data(data, filename):
    nlp = spacy.blank("en")
    db = DocBin()
    
    for text, annotations in data:
        doc = nlp.make_doc(text)
        # Assign the 'cats' dictionary directly to the doc object
        doc.cats = annotations["cats"]
        db.add(doc)
        
    db.to_disk(filename)
    print(f"Saved {len(data)} textcat examples to {filename}")

convert_textcat_data(TEXTCAT_TRAIN, "textcat_train.spacy")


<br><br>

---

<br><br>


## 🔄 Training the TextCat Model

Just like NER, we use the command line to train. But when generating the config, we specify `--pipeline textcat` instead of `ner`.


In [ ]:
"""
Step 1: Generate the base config for a TextCat model
!python -m spacy init config config.cfg --lang en --pipeline textcat --optimize efficiency

Step 2: Train the model
!python -m spacy train config.cfg --output ./textcat_models --paths.train ./textcat_train.spacy --paths.dev ./textcat_dev.spacy
"""


<br><br>

---

<br><br>


## 📊 Evaluating TextCat Performance

When you look at the training output for TextCat, you will see a few different metrics:

- **Accuracy**: The percentage of documents where the model guessed the exact right category. (If a dataset is 99% HAM and 1% SPAM, a model that *always* guesses HAM will have 99% accuracy, but it's a useless model!)
- **Macro F1**: Calculates the F1-Score for each category independently, and then averages them together. This is much better for unbalanced datasets (like the 99% HAM example).

### Confusion Matrix
A confusion matrix is a table that shows where the model is making mistakes. 
For example, if you have three categories (`BILLING`, `TECH`, `SALES`), the matrix will show you how many times a `TECH` ticket was accidentally classified as a `BILLING` ticket. spaCy makes it easy to generate these using external tools like `scikit-learn` based on the predicted `doc.cats`.
